In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, to_timestamp, explode
from pyspark.sql.types import ArrayType, StringType, StructType, StructField, LongType, BooleanType

In [2]:
spark = SparkSession.builder \
    .appName("inspect-bronze") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minio_admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minio_password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262"
    ) \
    .getOrCreate()

## Klines exploration

Analysis of strucure of data excracted from the Binance API.

Data are partitioned per ingestion date and then by symbol of the crypto currency inside MinIo bronze layer.

In [3]:
df = spark.read.parquet("s3a://bronze/binance/klines/ingestion_ts=20260520_09/symbol=ADAUSDT")
df.createOrReplaceTempView("klines")

Bronze payload contains a batch of Binance klines encoded as an array-of-arrays JSON structure.

In [4]:
result = spark.sql("""
    SELECT 
        bronze_source_data
    FROM klines
""")
result.show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [5]:
df.printSchema()
df.show(1, truncate=False)

root
 |-- bronze_source_data: string (nullable = true)
 |-- ingeted_at: timestamp (nullable = true)
 |-- ingestion_ts: string (nullable = true)
 |-- source: string (nullable = true)
 |-- data_type: string (nullable = true)
 |-- symbol: string (nullable = true)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Parse nested JSON into Spark Arrays.

In [9]:
schema = ArrayType(ArrayType(StringType()))
df2 = df.withColumn(
    "arr",
    from_json(col("bronze_source_data"), schema=schema)
)

Convert batch-oriented payload into one row per candlestick.

In [13]:
df_exploded = df2.withColumn(
    "kline",
    explode("arr")
)

Normalize Binance positional fields into typed columns.

In [11]:
df_silver_like = df_exploded.select(
    col("symbol"),
    (col("kline")[0] / 1000).cast("timestamp").alias("open_time"), # Binance uses milliseconds
    col("kline")[1].cast("double").alias("open"),
    col("kline")[2].cast("double").alias("high"),
    col("kline")[3].cast("double").alias("low"),
    col("kline")[4].cast("double").alias("close"),
    col("kline")[5].cast("double").alias("volume"),
)

For silver layer build just one table with all symbols together.

## Trades exploration

In [15]:
df_trades = spark.read.parquet("s3a://bronze/binance/trades/ingestion_ts=20260520_09/symbol=ADAUSDT")
df_trades.createOrReplaceTempView("trades")

In [16]:
result = spark.sql("""
    SELECT 
        bronze_source_data
    FROM trades
""")
result.show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [17]:
df_trades.printSchema()
df_trades.show(1, truncate=False)

root
 |-- bronze_source_data: string (nullable = true)
 |-- ingeted_at: timestamp (nullable = true)
 |-- ingestion_ts: string (nullable = true)
 |-- source: string (nullable = true)
 |-- data_type: string (nullable = true)
 |-- symbol: string (nullable = true)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Structure is a list of dictionaries.

In [20]:
trade_schema = ArrayType(
    StructType([
        StructField("id", LongType()),
        StructField("price", StringType()),
        StructField("qty", StringType()),
        StructField("quoteQty", StringType()),
        StructField("time", LongType()),
        StructField("isBuyerMaker", BooleanType()),
        StructField("isBestMatch", BooleanType())
    ])
)

In [22]:
df_trade_parsed = df_trades.withColumn(
    "trades_arr",
    from_json(col("bronze_source_data"), trade_schema)
)

In [23]:
df_trade_exploded = df_trade_parsed.withColumn(
    "trade",
    explode("trades_arr")
)

In [24]:
df_trade_silver_like = df_trade_exploded.select(
    col("symbol"),
    col("ingestion_ts"),

    col("trade.id").alias("trade_id"),
    col("trade.price").cast("double"),
    col("trade.qty").cast("double"),
    col("trade.quoteQty").cast("double"),
    (col("trade.time") / 1000).cast("timestamp").alias("trade_time"),
    col("trade.isBuyerMaker"),
    col("trade.isBestMatch")
)

In [25]:
df_trade_silver_like.show()

+-------+------------+---------+---------------------------+-------------------------+------------------------------+--------------------+------------+-----------+
| symbol|ingestion_ts| trade_id|CAST(trade.price AS DOUBLE)|CAST(trade.qty AS DOUBLE)|CAST(trade.quoteQty AS DOUBLE)|          trade_time|isBuyerMaker|isBestMatch|
+-------+------------+---------+---------------------------+-------------------------+------------------------------+--------------------+------------+-----------+
|ADAUSDT| 20260520_09|765011954|                     0.2496|                     31.6|                       7.88736|2026-05-20 09:16:...|       false|       true|
|ADAUSDT| 20260520_09|765011955|                     0.2496|                     28.8|                       7.18848|2026-05-20 09:16:...|       false|       true|
|ADAUSDT| 20260520_09|765011956|                     0.2496|                     30.7|                       7.66272|2026-05-20 09:16:...|       false|       true|
|ADAUSDT| 202605